In [24]:
import sys
sys.path.append('../src')
sys.path.append('../data')

In [25]:
import pandas as pd
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from seir_model import SEIRSimulator

In [26]:
# --- Load data ---
df = pd.read_csv('../data/data.csv')
# Load data

In [ ]:


observed_H = df["num_hospitalised"].values
observed_ICU = df["num_critical"].values
observed_D = df["num_fatalities"].values

n_days = len(df)
N = 1_000_000  # Total population
initial_state = [N - 10, 5, 5, 0, 0, 0, 0]  # [S, E, I, H, ICU, R, D]

# Fixed assumptions for latent delays
incubation_mean = 3.0   # E -> I
symptom_duration = 6.0  # I -> H or R

In [ ]:
# 📘 model_building.ipynb





# Load data
data = pd.read_csv("data.csv")
T = len(data)
N = 1_000_000

# Initial conditions
I0 = data.loc[0, "num_positive"]
H0 = data.loc[0, "num_hospitalised"]
ICU0 = data.loc[0, "num_critical"]
D0 = data.loc[0, "num_fatalities"]
E0 = 100
R0 = 0
S0 = N - E0 - I0 - H0 - ICU0 - R0 - D0
initial_state = np.array([S0, E0, I0, H0, ICU0, R0, D0])

with pm.Model() as model:
    # Priors (either deterministic or stochastic)
    beta = pm.Uniform("beta", 0.0, 1.0)
    sigma = pm.Uniform("sigma", 0.1, 1.0)  # incubation inverse
    gamma_I = pm.Uniform("gamma_I", 0.05, 0.5)  # recovery from I
    h_rate = pm.Uniform("h_rate", 0.01, 0.5)  # hospitalization rate
    gamma_H = pm.Uniform("gamma_H", 0.01, 0.5)
    icu_rate = pm.Uniform("icu_rate", 0.01, 0.5)
    gamma_ICU = pm.Uniform("gamma_ICU", 0.01, 0.5)
    mu = pm.Uniform("mu", 0.01, 0.3)

    # Define deterministic SEIR system
def seir_step(state, beta, sigma, gamma_I, h_rate, gamma_H, icu_rate, gamma_ICU, mu):
    S, E, I, H, ICU, R, D = state
    N = S + E + I + H + ICU + R + D

    new_exposed = beta * S * I / N
    new_infectious = sigma * E
    new_recovered_I = gamma_I * I
    new_hospitalized = h_rate * (I - new_recovered_I)

    new_recovered_H = gamma_H * H
    new_icu = icu_rate * (H - new_recovered_H)

    new_recovered_ICU = gamma_ICU * ICU
    new_dead = mu * (ICU - new_recovered_ICU)

    S = S - new_exposed
    E = E + new_exposed - new_infectious
    I = I + new_infectious - new_recovered_I - new_hospitalized
    H = H + new_hospitalized - new_recovered_H - new_icu
    ICU = ICU + new_icu - new_recovered_ICU - new_dead
    R = R + new_recovered_I + new_recovered_H + new_recovered_ICU
    D = D + new_dead

    return (S, E, I, H, ICU, R, D)

state = at.as_tensor_variable(initial_state)
results, _ = aesara.scan(
    fn=seir_step,
    sequences=[],
    outputs_info=[state],
    non_sequences=[beta, sigma, gamma_I, h_rate, gamma_H, icu_rate, gamma_ICU, mu],
    n_steps=T - 1,
)

# Observations (only I is partially observed)
I_series = results[:, 2]

# Adjusted for test sensitivity (~75%) and testing rate (~50% of symptomatics)
expected_positive = 0.5 * 0.75 * I_series
obs = pm.Normal("obs_I", mu=expected_positive, sigma=10, observed=data["num_positive"].values[1:])


TypeError: type TensorVariable doesn't define __round__ method